# 💳 Credit Card Fraud Detector
### Real-Time Fraud Detection using Random Forest + Imbalanced Data Handling

**Project:** Machine Learning / B.Tech Mini Project

**Goal:** Predict whether a credit-card transaction is legitimate (`0`) or fraudulent (`1`).

**Tech:** Python, Pandas, NumPy, Scikit-learn, Imbalanced-learn, Matplotlib, Seaborn, Random Forest.


## 📌 Dataset
This notebook expects the Kaggle **Credit Card Fraud Detection** dataset as `creditcard.csv`.

Upload the CSV using the upload cell below. The dataset normally contains a `Class` column where `0 = legitimate` and `1 = fraud`.


In [ ]:
# Install required libraries
!pip -q install imbalanced-learn


In [ ]:
# Load the dataset directly from GitHub (no manual upload needed)
# This GitHub Gist contains the standard Credit Card Fraud Detection dataset.
DATA_URL = 'https://gist.githubusercontent.com/MohammedAzarudeenBilal/820ffb6cdc19bde65e334845ce7dcf61/raw/credit%20card%20fraud%20detection%20dataset.csv'

df = pd.read_csv(DATA_URL)
print('Dataset loaded directly from GitHub.')
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE

print('Libraries imported successfully.')


In [ ]:
# Dataset was loaded from GitHub in the previous cell.
print('Dataset shape:', df.shape)
display(df.head())


In [ ]:
# Basic dataset inspection
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isnull().sum().sum())

print('\nDuplicate rows:', df.duplicated().sum())


In [ ]:
# Remove duplicate transactions if present
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate rows.')

# Verify target column
if 'Class' not in df.columns:
    raise ValueError("Target column 'Class' was not found. Rename your fraud-label column to 'Class'.")


In [ ]:
# Explore class distribution
class_counts = df['Class'].value_counts().sort_index()
print(class_counts)
print('\nPercentage distribution:')
print((df['Class'].value_counts(normalize=True) * 100).round(4))

plt.figure(figsize=(7,4))
sns.countplot(data=df, x='Class')
plt.title('Legitimate vs Fraudulent Transactions')
plt.xlabel('Class (0 = Legitimate, 1 = Fraud)')
plt.ylabel('Number of Transactions')
plt.show()


In [ ]:
# Optional numeric exploration
display(df.describe().T)

if 'Amount' in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x='Class', y='Amount')
    plt.title('Transaction Amount by Class')
    plt.ylim(0, df['Amount'].quantile(0.99))
    plt.show()


## 1️⃣ Prepare Features and Target
We keep `Class` as the target. `Time` and `Amount` are retained when present because they can carry useful transaction information.


In [ ]:
X = df.drop(columns=['Class'])
y = df['Class'].astype(int)

# Ensure all feature columns are numeric
non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric:
    raise ValueError(f'Non-numeric feature columns found: {non_numeric}. Encode them before training.')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training set:', X_train.shape)
print('Testing set :', X_test.shape)
print('\nTraining class distribution before SMOTE:')
print(y_train.value_counts())


## 2️⃣ Handle Class Imbalance with SMOTE
**Important:** SMOTE is applied only to the training set. The test set remains untouched so evaluation represents unseen real-world data.


In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('Before SMOTE:')
print(y_train.value_counts())

print('\nAfter SMOTE:')
print(pd.Series(y_train_smote).value_counts())


In [ ]:
# Visualize balanced training classes
plt.figure(figsize=(7,4))
sns.countplot(x=y_train_smote)
plt.title('Training Data After SMOTE')
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.show()


## 3️⃣ Train Random Forest Model
The model uses class-balanced training data generated by SMOTE. `n_jobs=-1` uses available CPU cores in Colab.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_smote, y_train_smote)
print('Random Forest training completed.')


In [ ]:
# Predictions on the untouched test set
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print(f'PR-AUC   : {pr_auc:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'], zero_division=0))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], '--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
# Precision-Recall curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(7,5))
plt.plot(recall_curve, precision_curve, label=f'PR-AUC = {pr_auc:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()


## 4️⃣ Feature Importance
Random Forest can estimate how much each feature contributes to the model's decisions.


In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(20).to_frame('Importance'))

plt.figure(figsize=(9,6))
importance.head(15).sort_values().plot(kind='barh')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.show()


## 5️⃣ Real-Time Transaction Fraud Prediction
The function below accepts a transaction as a dictionary using the exact feature columns from the training dataset. It returns a fraud probability and a decision.


In [ ]:
FEATURE_COLUMNS = X.columns.tolist()

def predict_transaction(transaction, threshold=0.50):
    """Predict fraud for one transaction.
    transaction must contain every feature used during model training.
    threshold controls the probability above which a transaction is flagged.
    """
    row = pd.DataFrame([transaction], columns=FEATURE_COLUMNS)
    missing = [c for c in FEATURE_COLUMNS if c not in transaction]
    if missing:
        raise ValueError(f'Missing feature(s): {missing}')
    if row.isnull().any().any():
        raise ValueError('Transaction contains missing values.')

    probability = float(rf_model.predict_proba(row)[0, 1])
    prediction = int(probability >= threshold)

    return {
        'prediction': prediction,
        'label': 'FRAUD' if prediction == 1 else 'LEGITIMATE',
        'fraud_probability': probability,
        'action': 'BLOCK / ALERT' if prediction == 1 else 'APPROVE'
    }


In [ ]:
# Demo: use a real transaction from the test set as an example.
# This demonstrates the prediction pipeline without manually typing hundreds of V-columns.
sample_transaction = X_test.iloc[0].to_dict()
result = predict_transaction(sample_transaction, threshold=0.50)
print(result)


In [ ]:
# Test several transactions and show their predicted probabilities
demo_rows = X_test.head(10).copy()
demo_probabilities = rf_model.predict_proba(demo_rows)[:, 1]
demo_predictions = (demo_probabilities >= 0.50).astype(int)

demo_output = pd.DataFrame({
    'Actual': y_test.loc[demo_rows.index].values,
    'Fraud_Probability': demo_probabilities,
    'Prediction': demo_predictions,
    'Decision': np.where(demo_predictions == 1, 'BLOCK / ALERT', 'APPROVE')
})
display(demo_output)


## 6️⃣ Save the Trained Model
The model can be saved and later loaded into a Flask/FastAPI application for a real-time API.


In [ ]:
import joblib

MODEL_FILE = 'credit_card_fraud_random_forest.pkl'
joblib.dump({
    'model': rf_model,
    'features': FEATURE_COLUMNS,
    'threshold': 0.50
}, MODEL_FILE)

print(f'Model saved as: {MODEL_FILE}')


In [ ]:
# Download the trained model to your computer
from google.colab import files
files.download(MODEL_FILE)


## 7️⃣ Optional: Download Evaluation Results
Creates a CSV containing actual labels, predictions, and fraud probabilities.


In [ ]:
results = X_test.copy()
results['Actual_Class'] = y_test.values
results['Fraud_Probability'] = y_prob
results['Predicted_Class'] = y_pred
results['Decision'] = np.where(y_pred == 1, 'BLOCK / ALERT', 'APPROVE')

RESULT_FILE = 'fraud_detection_results.csv'
results.to_csv(RESULT_FILE, index=False)
print(f'Saved: {RESULT_FILE}')
files.download(RESULT_FILE)


## 🎓 Project Conclusion
The system preprocesses credit-card transaction data, addresses severe class imbalance using SMOTE, trains a Random Forest classifier, evaluates it with fraud-focused metrics, visualizes model performance, identifies important features, and provides a reusable transaction-level prediction function.

**For a real banking deployment:** this notebook is a prototype. A production system should additionally use secure APIs, authentication, monitoring, drift detection, calibrated thresholds, human review workflows, and carefully designed false-positive/false-negative controls.
